In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (0-100% of gait cycle)
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']

# Feature sets for the three sensor configurations
IMU_COLUMNS = [
    'IMU101_v0', 'IMU101_v1',
    'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
]
PRESSURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]
CONFIGS = {
    'IMU':          IMU_COLUMNS,
    'Pressure':     PRESSURE_COLUMNS,
    'IMU+Pressure': IMU_COLUMNS + PRESSURE_COLUMNS,
}

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

MODEL_TYPES = ['lstm', 'gru', 'cnn']
RESULTS_DIR = 'gait_hardware_paper_results'
RESULTS_CSV = os.path.join(RESULTS_DIR, 'personalized_results.csv')


def get_subject_id(file_path):
    """Subject = top-level folder under ROOT_DIR (e.g. 'Andy_Dynamic')."""
    rel = os.path.relpath(file_path, ROOT_DIR)
    return rel.split(os.sep)[0]


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_all_raw():
    """Load every cycle once at native length. Returns list of (df, label, subject)."""
    raw = []
    if not os.path.exists(ROOT_DIR):
        print(f"ROOT_DIR not found: {ROOT_DIR}")
        return raw

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                raw.append((df, label, get_subject_id(file_path)))

    return raw


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def prepare_subject_matrix(subject_rows, feature_cols):
    """From this subject's raw rows, select features, resample, and stack.
    Returns X (n,TARGET_LEN,F), y_str (n,), or (None, None) if unusable."""
    X, y = [], []
    for df, label, _ in subject_rows:
        if any(c not in df.columns for c in feature_cols):
            continue
        vals = df[feature_cols].values
        if vals.shape[0] < 2 or vals.shape[1] == 0:
            continue
        X.append(resample_cycle(vals))
        y.append(label)
    if len(X) == 0:
        return None, None
    return np.array(X), np.array(y)


def run_personalized():
    raw = load_all_raw()
    if not raw:
        print("No cycles loaded.")
        return

    subjects = sorted({r[2] for r in raw})
    print(f"Detected {len(subjects)} subjects: {subjects}\n")

    rows = []  # one row per (subject, config, model)

    for subj in subjects:
        subj_rows = [r for r in raw if r[2] == subj]
        print(f"\n{'='*55}\n=== Subject: {subj}  ({len(subj_rows)} cycles) ===\n{'='*55}")

        for cfg_name, feature_cols in CONFIGS.items():
            X, y_str = prepare_subject_matrix(subj_rows, feature_cols)
            if X is None:
                print(f"  [{cfg_name}] no usable cycles, skipping")
                continue

            # Encode labels for THIS subject (a subject may lack a class)
            le = LabelEncoder()
            y_int = le.fit_transform(y_str)
            class_names = le.classes_
            num_classes = len(class_names)
            y_cat = to_categorical(y_int, num_classes=num_classes)

            n = X.shape[0]
            # Need at least a few cycles per class to split; require >= 2 per class
            unique, counts = np.unique(y_int, return_counts=True)
            if n < 10 or counts.min() < 2 or num_classes < 2:
                print(f"  [{cfg_name}] too few cycles (n={n}, min/class={counts.min()}), skipping")
                continue

            # Within-subject random 80/20 split, stratified by class where possible
            try:
                Xtr, Xte, ytr, yte, ytr_int, yte_int = train_test_split(
                    X, y_cat, y_int, train_size=TRAIN_RATIO,
                    random_state=42, shuffle=True, stratify=y_int)
            except ValueError:
                # stratify fails if a class has too few samples; fall back
                Xtr, Xte, ytr, yte, ytr_int, yte_int = train_test_split(
                    X, y_cat, y_int, train_size=TRAIN_RATIO,
                    random_state=42, shuffle=True)

            seq_len = X.shape[1]
            num_features = X.shape[2]

            for mtype in MODEL_TYPES:
                tf.keras.backend.clear_session()
                model = build_model(mtype, seq_len, num_features, num_classes)
                model.fit(
                    Xtr, ytr,
                    epochs=EPOCHS, batch_size=BATCH_SIZE,
                    callbacks=[EarlyStopping(monitor='loss', patience=5,
                                             restore_best_weights=True, verbose=0)],
                    verbose=0
                )
                pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
                acc = accuracy_score(yte_int, pred)
                f1m = f1_score(yte_int, pred, average='macro', zero_division=0)
                f1w = f1_score(yte_int, pred, average='weighted', zero_division=0)

                print(f"  [{cfg_name:12s} {mtype.upper():4s}] "
                      f"n_train={len(ytr_int):4d} n_test={len(yte_int):3d}  acc={acc:.4f}")

                rows.append({
                    'subject':   subj,
                    'config':    cfg_name,
                    'model':     mtype,
                    'n_cycles':  n,
                    'n_train':   len(ytr_int),
                    'n_test':    len(yte_int),
                    'n_classes': num_classes,
                    'accuracy':  round(acc, 4),
                    'macro_f1':  round(f1m, 4),
                    'weighted_f1': round(f1w, 4),
                })

    if not rows:
        print("\nNo results produced.")
        return

    df = pd.DataFrame(rows)
    os.makedirs(RESULTS_DIR, exist_ok=True)   # create folder if it doesn't exist
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n\nWrote {len(df)} rows to {RESULTS_CSV}")
    print("\nQuick pivot (accuracy by subject x config x model):")
    with pd.option_context('display.max_rows', None, 'display.width', 120):
        print(df.pivot_table(index='subject', columns=['config', 'model'],
                             values='accuracy'))


if __name__ == '__main__':
    tf.get_logger().setLevel('ERROR')
    run_personalized()

2026-08-02 21:09:05.703810: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785719345.717802 4154105 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785719345.722095 4154105 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785719345.734459 4154105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785719345.734479 4154105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785719345.734482 4154105 computation_placer.cc:177] computation placer alr

Detected 10 subjects: ['Andy_Dynamic', 'Ankan_Dynamic', 'David_Dynamic', 'Hrithik_Dynamic', 'JJ_Dynamic', 'Mustafa_Dynamic', 'Rezoan_Dynamic', 'Sudipta_Dynamic', 'Tianjun_Dynamic', 'Zongwei_Dynamic']


=== Subject: Andy_Dynamic  (424 cycles) ===


I0000 00:00:1785719362.996421 4154105 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 45848 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1785719362.998302 4154105 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 46677 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:25:00.0, compute capability: 8.6
I0000 00:00:1785719362.999611 4154105 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 46677 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1785719363.001043 4154105 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 46677 MB memory:  -> device: 3, name: NVIDIA RTX A6000, pci bus id: 0000:c1:00.0, compute capability: 8.6
/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWar

  [IMU          LSTM] n_train= 339 n_test= 85  acc=1.0000


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

  [IMU+Pressure LSTM] n_train= 544 n_test=137  acc=0.9927


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  [IMU+Pressure GRU ] n_train= 544 n_test=137  acc=0.9927


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  [IMU+Pressure CNN ] n_train= 544 n_test=137  acc=1.0000


Wrote 90 rows to gait_hardware_paper_results/personalized_results.csv

Quick pivot (accuracy by subject x config x model):
config              IMU                 IMU+Pressure                 Pressure                
model               cnn     gru    lstm          cnn     gru    lstm      cnn     gru    lstm
subject                                                                                      
Andy_Dynamic     0.8824  0.9647  1.0000       0.9882  0.9882  0.9882   0.9529  0.9647  0.9647
Ankan_Dynamic    0.9434  0.9811  0.9811       0.9811  0.9434  0.9340   0.9434  0.7830  0.9245
David_Dynamic    0.9804  0.9412  0.9216       0.9902  0.9608  0.9902   0.9902  0.9706  0.9804
Hrithik_Dynamic  1.0000  1.0000  0.9897       1.0000  0.9794  1.0000   0.9897  0.9691  0.9794
JJ_Dynamic       1.0000  0.9896  0.9896       0.9479  0.9896  0.9792   0.9583  0.9167  0.9583
Mustafa_Dynamic  0.9912  0.9823  1.0000       1.0000  1.0000  0.9

In [2]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (0-100% of gait cycle)
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']

# Feature sets for the three sensor configurations
IMU_COLUMNS = [
    'IMU101_v0', 'IMU101_v1',
    'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
]
PRESSURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]
CONFIGS = {
    'IMU':          IMU_COLUMNS,
    'Pressure':     PRESSURE_COLUMNS,
    'IMU+Pressure': IMU_COLUMNS + PRESSURE_COLUMNS,
}

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

MODEL_TYPES = ['lstm', 'gru', 'cnn']
RESULTS_DIR = 'gait_hardware_paper_results'
RESULTS_CSV = os.path.join(RESULTS_DIR, 'personalized_results.csv')


def get_subject_id(file_path):
    """Subject = top-level folder under ROOT_DIR (e.g. 'Andy_Dynamic')."""
    rel = os.path.relpath(file_path, ROOT_DIR)
    return rel.split(os.sep)[0]


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_all_raw():
    """Load every cycle once at native length. Returns list of (df, label, subject)."""
    raw = []
    if not os.path.exists(ROOT_DIR):
        print(f"ROOT_DIR not found: {ROOT_DIR}")
        return raw

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                raw.append((df, label, get_subject_id(file_path)))

    return raw


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def prepare_subject_matrix(subject_rows, feature_cols):
    """From this subject's raw rows, select features, resample, and stack.
    Returns X (n,TARGET_LEN,F), y_str (n,), or (None, None) if unusable."""
    X, y = [], []
    for df, label, _ in subject_rows:
        if any(c not in df.columns for c in feature_cols):
            continue
        vals = df[feature_cols].values
        if vals.shape[0] < 2 or vals.shape[1] == 0:
            continue
        X.append(resample_cycle(vals))
        y.append(label)
    if len(X) == 0:
        return None, None
    return np.array(X), np.array(y)


def run_personalized():
    raw = load_all_raw()
    if not raw:
        print("No cycles loaded.")
        return

    subjects = sorted({r[2] for r in raw})
    print(f"Detected {len(subjects)} subjects: {subjects}\n")

    rows = []  # one row per (subject, config, model)

    for subj in subjects:
        subj_rows = [r for r in raw if r[2] == subj]
        print(f"\n{'='*55}\n=== Subject: {subj}  ({len(subj_rows)} cycles) ===\n{'='*55}")

        for cfg_name, feature_cols in CONFIGS.items():
            X, y_str = prepare_subject_matrix(subj_rows, feature_cols)
            if X is None:
                print(f"  [{cfg_name}] no usable cycles, skipping")
                continue

            # Encode labels for THIS subject (a subject may lack a class)
            le = LabelEncoder()
            y_int = le.fit_transform(y_str)
            class_names = le.classes_
            num_classes = len(class_names)
            y_cat = to_categorical(y_int, num_classes=num_classes)

            n = X.shape[0]
            # Need at least a few cycles per class to split; require >= 2 per class
            unique, counts = np.unique(y_int, return_counts=True)
            if n < 10 or counts.min() < 2 or num_classes < 2:
                print(f"  [{cfg_name}] too few cycles (n={n}, min/class={counts.min()}), skipping")
                continue

            # Within-subject random 80/20 split, stratified by class where possible
            try:
                Xtr, Xte, ytr, yte, ytr_int, yte_int = train_test_split(
                    X, y_cat, y_int, train_size=TRAIN_RATIO,
                    random_state=42, shuffle=True, stratify=y_int)
            except ValueError:
                # stratify fails if a class has too few samples; fall back
                Xtr, Xte, ytr, yte, ytr_int, yte_int = train_test_split(
                    X, y_cat, y_int, train_size=TRAIN_RATIO,
                    random_state=42, shuffle=True)

            seq_len = X.shape[1]
            num_features = X.shape[2]

            for mtype in MODEL_TYPES:
                tf.keras.backend.clear_session()
                model = build_model(mtype, seq_len, num_features, num_classes)
                model.fit(
                    Xtr, ytr,
                    epochs=EPOCHS, batch_size=BATCH_SIZE,
                    callbacks=[EarlyStopping(monitor='loss', patience=5,
                                             restore_best_weights=True, verbose=0)],
                    verbose=0
                )
                pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
                acc = accuracy_score(yte_int, pred)
                f1m = f1_score(yte_int, pred, average='macro', zero_division=0)
                f1w = f1_score(yte_int, pred, average='weighted', zero_division=0)

                print(f"  [{cfg_name:12s} {mtype.upper():4s}] "
                      f"n_train={len(ytr_int):4d} n_test={len(yte_int):3d}  acc={acc:.4f}")

                rows.append({
                    'subject':   subj,
                    'config':    cfg_name,
                    'model':     mtype,
                    'n_cycles':  n,
                    'n_train':   len(ytr_int),
                    'n_test':    len(yte_int),
                    'n_classes': num_classes,
                    'accuracy':  round(acc, 4),
                    'macro_f1':  round(f1m, 4),
                    'weighted_f1': round(f1w, 4),
                })

    if not rows:
        print("\nNo results produced.")
        return

    df = pd.DataFrame(rows)
    os.makedirs(RESULTS_DIR, exist_ok=True)   # create folder if it doesn't exist
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n\nWrote {len(df)} rows to {RESULTS_CSV}")
    print("\nQuick pivot (accuracy by subject x config x model):")
    with pd.option_context('display.max_rows', None, 'display.width', 120):
        print(df.pivot_table(index='subject', columns=['config', 'model'],
                             values='accuracy'))


if __name__ == '__main__':
    tf.get_logger().setLevel('ERROR')
    run_personalized()

Detected 10 subjects: ['Andy_Dynamic', 'Ankan_Dynamic', 'David_Dynamic', 'Hrithik_Dynamic', 'JJ_Dynamic', 'Mustafa_Dynamic', 'Rezoan_Dynamic', 'Sudipta_Dynamic', 'Tianjun_Dynamic', 'Zongwei_Dynamic']


=== Subject: Andy_Dynamic  (424 cycles) ===


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  [IMU          LSTM] n_train= 339 n_test= 85  acc=0.9412


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  [IMU          GRU ] n_train= 339 n_test= 85  acc=0.9529


...

  [IMU+Pressure LSTM] n_train= 544 n_test=137  acc=1.0000


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  [IMU+Pressure GRU ] n_train= 544 n_test=137  acc=1.0000


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  [IMU+Pressure CNN ] n_train= 544 n_test=137  acc=1.0000


Wrote 90 rows to gait_hardware_paper_results/personalized_results.csv

Quick pivot (accuracy by subject x config x model):
config              IMU                 IMU+Pressure                 Pressure                
model               cnn     gru    lstm          cnn     gru    lstm      cnn     gru    lstm
subject                                                                                      
Andy_Dynamic     0.9765  0.9529  0.9412       0.9882  0.9294  0.9529   0.9412  0.9529  0.9529
Ankan_Dynamic    0.9623  0.9717  0.9623       0.9811  0.9811  0.9811   0.9528  0.8491  0.9717
David_Dynamic    0.9706  0.9510  0.8529       1.0000  0.9608  0.9804   0.9902  0.9902  0.9804
Hrithik_Dynamic  1.0000  1.0000  1.0000       1.0000  0.9897  1.0000   0.9897  0.9485  0.8866
JJ_Dynamic       1.0000  0.9583  0.9896       0.9688  0.9896  0.9896   0.9479  0.9167  0.9479
Mustafa_Dynamic  1.0000  0.9912  0.9735       1.0000  1.0000  1.0